# Tutorial: eQTL Analysis with `SAIGE-QTL` using cellink
This tutorial demonstrates how to perform eQTL analysis using `SAIGE-QTL` through the cellink package. `SAIGE-QTL` uses a Poisson mixed model that accounts for sample relatedness and is particularly well-suited for single-cell eQTL analysis. It also accounts for sample relatedness through genetic relationship matrices, handles both common and rare variant analysis and supports cell-level and individual-level covariates. This notebook assumes familiarity with single-cell data processing and basic statistical genetics concepts. The cellink package provides convenient wrapper functions that handle data preparation and formatting for `SAIGE-QTL`. For the installation please refer to the docs of [`SAIGE-QTL`](https://weizhou0.github.io/SAIGE-QTL-doc/docs/Installation.html). We recommend using the provided [Docker container](https://hub.docker.com/r/wzhou88/saigeqtl).

SAIGE-QTL analysis consists of three steps:
- Step 1: Fit null Poisson mixed model (one per gene)
- Step 2: Perform association tests (single-variant or set-based)
- Step 3 (optional): Calculate gene-level p-values using ACAT

SAIGE-QTL has a range of further options, such as set-based analysis, genome-wide analysis with LOCO or conditional analysis, which are alll supported in the `cellink` wrapper. For more details please however refer to the docs of [`SAIGE-QTL`](https://weizhou0.github.io/SAIGE-QTL-doc/docs/Installation.html).

## Environment Setup

In [1]:
import numpy as np
import scanpy as sc

from cellink.resources import get_dummy_onek1k
from cellink.tl.external import run_saigeqtl, configure_saigeqtl_runner

# Analysis parameters
n_gpcs = 20
n_epcs = 15
chrom = 22
cis_window = 500_000
cell_type = "CD8 Naive"
celltype_key = "predicted.celltype.l2"

site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


## Configure SAIGE-QTL Runner
The runner handles execution across different environments (local, docker, singularity):

In [2]:
runner = configure_saigeqtl_runner("saigeqtl_docker")

## Load and Prepare Data

In [3]:
hd = get_dummy_onek1k(verify_checksum=False)
print(f"Dataset shape: {hd.shape}")

hd.G.obsm["gPCs"] = hd.G.obsm["gPCs"][hd.G.obsm["gPCs"].columns[:n_gpcs]]

hd.aggregate(obs=["donor_id", "sex", "age"], func="first", add_to_obs=True)

hd = hd[..., hd.C.obs[celltype_key] == cell_type, :].copy()
print(f"After cell type filtering: {hd.shape}")

hd.G.obs["donor_sex"] = hd.G.obs["sex"]
hd.G.obs["donor_age"] = hd.G.obs["age"]

hd = hd.sel(G_var=hd.G.var.chrom == str(chrom), C_var=hd.C.var.chrom == str(chrom)).copy()
print(f"After chromosome {chrom} filtering: {hd.shape}")

hd = hd[:, (hd.G.var["pos"] > 41_900_000) & (hd.G.var["pos"] < 45_600_000), :, :].copy()

# Gene to test: SMDT1 (ENSG00000183172), one of the two strongest CD8 naive T cell cis-eQTLs on chromosome 22 (with FAM118A)
gene_counts = np.asarray(hd.C.X.sum(axis=0)).ravel()
gene_idx = hd.C.var_names.get_loc("ENSG00000183172")
gene_id = hd.C.var_names[gene_idx]
gene_symbol = hd.C.var["feature_name"].iloc[gene_idx]
print(f"Testing {gene_id} ({gene_symbol}): {int(gene_counts[gene_idx])} counts in {hd.C.n_obs} cells")

[2026-11-23 23:00:08,736] INFO:root: ~/cellink_data/dummy_onek1k/dummy_onek1k_dataset.hd.h5 already exists


[2026-11-23 23:00:08,736] WARNING:root: No checksum provided, skipping verification


[2026-11-23 23:00:16,230] INFO:root: Loaded dummy OneK1K dataset: (300, 146939, 125913, 34830)


Dataset shape: (300, 146939, 125913, 34830)


After cell type filtering: (294, 146939, 4859, 34830)


After chromosome 22 filtering: (294, 136776, 4859, 888)
Testing ENSG00000183172 (SMDT1_ENSG00000183172): 5260 counts in 4859 cells


## Analysis Workflows
### 1. Basic cis-eQTL Analysis (Single-Variant Tests, Steps 1 + 2)
The most common use case: fit null model and run association tests.

In [4]:
results = run_saigeqtl(
    hd,
    gene_col=gene_id,  # expressed gene chosen above
    prefix="saigeqtl_complete",
    steps=[1, 2],
    mode="cis",
    analysis_type="single_variant",
    window=cis_window,
    min_mac=20,
    sample_covariates=["gPCs"],
    use_grm_to_fit_null=False,
    overwrite_variance_ratio_file=True,
    run=True,
)

results.head()

[2026-11-23 23:00:20,776] INFO:cellink.io._export: Donor sex coded 0/1 is read as 0 = female, 1 = male (as in get_onek1k) and written as PLINK codes 2 (female) and 1 (male). Pass `donor_sex_map={0: 0, 1: 1}` if 0 means unknown.


[2026-11-23 23:00:21,060] INFO:cellink.io._export: Wrote PLINK 1 binary files saigeqtl_complete.bed/.bim/.fam (294 x 17569)


[2026-11-23 23:00:21,115] INFO:cellink.tl.external._saigeqtl: Running Step 1: Fitting null model


[2026-11-23 23:00:21,119] INFO:cellink.tl._runner: Executing: docker run --rm -v .:/data -v ~/cellink_data:/cellink_data -w /data wzhou88/saigeqtl /app/.pixi/envs/default/bin/Rscript /app/extdata/step1_fitNULLGLMM_qtl.R --phenoFile=saigeqtl_complete_phenotype.txt --phenoCol=ENSG00000183172 --sampleIDColinphenoFile=IND_ID --traitType=count --plinkFile=saigeqtl_complete --outputPrefix=saigeqtl_complete --useGRMtoFitNULL=FALSE --IsOverwriteVarianceRatioFile=TRUE --covarColList=sex,age,gPCs_0,gPCs_1,gPCs_2,gPCs_3,gPCs_4,gPCs_5,gPCs_6,gPCs_7,gPCs_8,gPCs_9,gPCs_10,gPCs_11,gPCs_12,gPCs_13,gPCs_14,gPCs_15,gPCs_16,gPCs_17,gPCs_18,gPCs_19 --sampleCovarColList=sex,age,gPCs_0,gPCs_1,gPCs_2,gPCs_3,gPCs_4,gPCs_5,gPCs_6,gPCs_7,gPCs_8,gPCs_9,gPCs_10,gPCs_11,gPCs_12,gPCs_13,gPCs_14,gPCs_15,gPCs_16,gPCs_17,gPCs_18,gPCs_19


[2026-11-23 23:00:30,878] INFO:cellink.tl._runner: WARNING: ignoring environment value of R_HOME
R version 4.5.2 (2025-10-31)
Platform: x86_64-conda-linux-gnu
Running under: Ubuntu 20.04.6 LTS

Matrix products: default
BLAS/LAPACK: /app/.pixi/envs/default/lib/libopenblasp-r0.3.30.so;  LAPACK version 3.12.0

locale:
 [1] LC_CTYPE=C.UTF-8       LC_NUMERIC=C           LC_TIME=C.UTF-8       
 [4] LC_COLLATE=C.UTF-8     LC_MONETARY=C.UTF-8    LC_MESSAGES=C.UTF-8   
 [7] LC_PAPER=C.UTF-8       LC_NAME=C              LC_ADDRESS=C          
[10] LC_TELEPHONE=C         LC_MEASUREMENT=C.UTF-8 LC_IDENTIFICATION=C   

time zone: NA
tzcode source: system (glibc)

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
[1] SAIGEQTL_0.3.5.2 optparse_1.7.5  

loaded via a namespace (and not attached):
 [1] compiler_4.5.2        Matrix_1.7-4          tools_4.5.2          
 [4] Rcpp_1.1.1            getopt_1.20.4         RcppNumerical_0

[2026-11-23 23:00:30,880] WARNING:cellink.tl._runner: Loading required package: optparse
Loading required package: RhpcBLASctl
Loading required package: fastSave



[2026-11-23 23:00:30,880] INFO:cellink.tl.external._saigeqtl: Running Step 2: Association tests


[2026-11-23 23:00:30,884] INFO:cellink.tl._runner: Executing: docker run --rm -v .:/data -v ~/cellink_data:/cellink_data -w /data wzhou88/saigeqtl /app/.pixi/envs/default/bin/Rscript /app/extdata/step2_tests_qtl.R --bedFile=saigeqtl_complete.bed --bimFile=saigeqtl_complete.bim --famFile=saigeqtl_complete.fam --GMMATmodelFile=saigeqtl_complete.rda --varianceRatioFile=saigeqtl_complete.varianceRatio.txt --SAIGEOutputFile=saigeqtl_complete_results.txt --minMAC=20 --LOCO=FALSE --rangestoIncludeFile=saigeqtl_complete_ranges.txt


[2026-11-23 23:00:37,458] INFO:cellink.tl._runner: WARNING: ignoring environment value of R_HOME
R version 4.5.2 (2025-10-31)
Platform: x86_64-conda-linux-gnu
Running under: Ubuntu 20.04.6 LTS

Matrix products: default
BLAS/LAPACK: /app/.pixi/envs/default/lib/libopenblasp-r0.3.30.so;  LAPACK version 3.12.0

locale:
 [1] LC_CTYPE=C.UTF-8       LC_NUMERIC=C           LC_TIME=C.UTF-8       
 [4] LC_COLLATE=C.UTF-8     LC_MONETARY=C.UTF-8    LC_MESSAGES=C.UTF-8   
 [7] LC_PAPER=C.UTF-8       LC_NAME=C              LC_ADDRESS=C          
[10] LC_TELEPHONE=C         LC_MEASUREMENT=C.UTF-8 LC_IDENTIFICATION=C   

time zone: NA
tzcode source: system (glibc)

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
[1] RhpcBLASctl_0.23-42 SAIGEQTL_0.3.5.2    data.table_1.17.8  
[4] optparse_1.7.5     

loaded via a namespace (and not attached):
[1] compiler_4.5.2        Matrix_1.7-4          tools_4.5.2          
[4] Rcpp_1.1.1 

[2026-11-23 23:00:37,459] WARNING:cellink.tl._runner: Loading required package: RhpcBLASctl
Loading required package: furrr
Loading required package: future



,CHR,POS,MarkerID,Allele1,Allele2,AC_Allele2,AF_Allele2,MissingRate,BETA,SE,Tstat,var,p.value,p.value.NA,Is.SPA,N
0,22,41900699,22_41900699_T_C,T,C,46,0.078231,0,-0.515473,0.087553,-67.2462,130.455,6.442200e-09,3.918860e-09,True,294
1,22,41901918,22_41901918_G_A,G,A,46,0.078231,0,-0.515473,0.087553,-67.2462,130.455,6.442200e-09,3.918860e-09,True,294
2,22,41901964,22_41901964_T_C,T,C,46,0.078231,0,-0.515473,0.087553,-67.2462,130.455,6.442200e-09,3.918860e-09,True,294
3,22,41902161,22_41902161_C_T,C,T,45,0.076531,0,-0.517128,0.087725,-67.1978,129.944,6.191220e-09,3.748950e-09,True,294
4,22,41902597,22_41902597_C_T,C,T,46,0.078231,0,-0.515473,0.087553,-67.2462,130.455,6.442200e-09,3.918860e-09,True,294


### 2. Step-by-Step Analysis
Run each step independently for more control:
#### Step 1 Only: Fit Null Model

In [5]:
step1_results = run_saigeqtl(
    hd,
    gene_col=gene_id,
    prefix="gene1_step1",
    steps=[1],  # Only Step 1
    sample_covariates=["gPCs"],
    tol=0.00001,
    maxiter=20,
    use_grm_to_fit_null=False,
    overwrite_variance_ratio_file=True,
    run=True,
)

print(step1_results)

[2026-11-23 23:00:39,550] INFO:cellink.io._export: Donor sex coded 0/1 is read as 0 = female, 1 = male (as in get_onek1k) and written as PLINK codes 2 (female) and 1 (male). Pass `donor_sex_map={0: 0, 1: 1}` if 0 means unknown.


[2026-11-23 23:00:39,787] INFO:cellink.io._export: Wrote PLINK 1 binary files gene1_step1.bed/.bim/.fam (294 x 17569)


[2026-11-23 23:00:39,834] INFO:cellink.tl.external._saigeqtl: Running Step 1: Fitting null model


[2026-11-23 23:00:39,837] INFO:cellink.tl._runner: Executing: docker run --rm -v .:/data -v ~/cellink_data:/cellink_data -w /data wzhou88/saigeqtl /app/.pixi/envs/default/bin/Rscript /app/extdata/step1_fitNULLGLMM_qtl.R --phenoFile=gene1_step1_phenotype.txt --phenoCol=ENSG00000183172 --sampleIDColinphenoFile=IND_ID --traitType=count --plinkFile=gene1_step1 --outputPrefix=gene1_step1 --tol=1e-05 --useGRMtoFitNULL=FALSE --IsOverwriteVarianceRatioFile=TRUE --covarColList=sex,age,gPCs_0,gPCs_1,gPCs_2,gPCs_3,gPCs_4,gPCs_5,gPCs_6,gPCs_7,gPCs_8,gPCs_9,gPCs_10,gPCs_11,gPCs_12,gPCs_13,gPCs_14,gPCs_15,gPCs_16,gPCs_17,gPCs_18,gPCs_19 --sampleCovarColList=sex,age,gPCs_0,gPCs_1,gPCs_2,gPCs_3,gPCs_4,gPCs_5,gPCs_6,gPCs_7,gPCs_8,gPCs_9,gPCs_10,gPCs_11,gPCs_12,gPCs_13,gPCs_14,gPCs_15,gPCs_16,gPCs_17,gPCs_18,gPCs_19


[2026-11-23 23:00:45,107] INFO:cellink.tl._runner: WARNING: ignoring environment value of R_HOME
R version 4.5.2 (2025-10-31)
Platform: x86_64-conda-linux-gnu
Running under: Ubuntu 20.04.6 LTS

Matrix products: default
BLAS/LAPACK: /app/.pixi/envs/default/lib/libopenblasp-r0.3.30.so;  LAPACK version 3.12.0

locale:
 [1] LC_CTYPE=C.UTF-8       LC_NUMERIC=C           LC_TIME=C.UTF-8       
 [4] LC_COLLATE=C.UTF-8     LC_MONETARY=C.UTF-8    LC_MESSAGES=C.UTF-8   
 [7] LC_PAPER=C.UTF-8       LC_NAME=C              LC_ADDRESS=C          
[10] LC_TELEPHONE=C         LC_MEASUREMENT=C.UTF-8 LC_IDENTIFICATION=C   

time zone: NA
tzcode source: system (glibc)

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
[1] SAIGEQTL_0.3.5.2 optparse_1.7.5  

loaded via a namespace (and not attached):
 [1] compiler_4.5.2        Matrix_1.7-4          tools_4.5.2          
 [4] Rcpp_1.1.1            getopt_1.20.4         RcppNumerical_0

[2026-11-23 23:00:45,108] WARNING:cellink.tl._runner: Loading required package: optparse
Loading required package: RhpcBLASctl
Loading required package: fastSave



{'model_file': 'gene1_step1.rda', 'variance_ratio':           0            1  2
0  0.959333       sparse  1
1  0.230154          NaN  1
2  0.196514  null_noXadj  1
3  0.230154  null_sample  1}


#### Step 2 Only: Association Testing

In [6]:
step2_results = run_saigeqtl(
    hd,
    gene_col=gene_id,
    prefix="gene1_step1",  # Use same prefix as Step 1
    steps=[2],  # Only Step 2
    mode="cis",
    window=cis_window,
    min_mac=20,
    gmmat_model_file="gene1_step1.rda",  # From Step 1
    variance_ratio_file="gene1_step1.varianceRatio.txt",  # From Step 1
    run=True,
)

step2_results.head()

[2026-11-23 23:00:46,998] INFO:cellink.io._export: Donor sex coded 0/1 is read as 0 = female, 1 = male (as in get_onek1k) and written as PLINK codes 2 (female) and 1 (male). Pass `donor_sex_map={0: 0, 1: 1}` if 0 means unknown.


[2026-11-23 23:00:47,281] INFO:cellink.io._export: Wrote PLINK 1 binary files gene1_step1.bed/.bim/.fam (294 x 17569)


[2026-11-23 23:00:47,361] INFO:cellink.tl.external._saigeqtl: Running Step 2: Association tests


[2026-11-23 23:00:47,365] INFO:cellink.tl._runner: Executing: docker run --rm -v .:/data -v ~/cellink_data:/cellink_data -w /data wzhou88/saigeqtl /app/.pixi/envs/default/bin/Rscript /app/extdata/step2_tests_qtl.R --bedFile=gene1_step1.bed --bimFile=gene1_step1.bim --famFile=gene1_step1.fam --GMMATmodelFile=gene1_step1.rda --varianceRatioFile=gene1_step1.varianceRatio.txt --SAIGEOutputFile=gene1_step1_results.txt --minMAC=20 --LOCO=FALSE --rangestoIncludeFile=gene1_step1_ranges.txt


[2026-11-23 23:00:53,051] INFO:cellink.tl._runner: WARNING: ignoring environment value of R_HOME
R version 4.5.2 (2025-10-31)
Platform: x86_64-conda-linux-gnu
Running under: Ubuntu 20.04.6 LTS

Matrix products: default
BLAS/LAPACK: /app/.pixi/envs/default/lib/libopenblasp-r0.3.30.so;  LAPACK version 3.12.0

locale:
 [1] LC_CTYPE=C.UTF-8       LC_NUMERIC=C           LC_TIME=C.UTF-8       
 [4] LC_COLLATE=C.UTF-8     LC_MONETARY=C.UTF-8    LC_MESSAGES=C.UTF-8   
 [7] LC_PAPER=C.UTF-8       LC_NAME=C              LC_ADDRESS=C          
[10] LC_TELEPHONE=C         LC_MEASUREMENT=C.UTF-8 LC_IDENTIFICATION=C   

time zone: NA
tzcode source: system (glibc)

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
[1] RhpcBLASctl_0.23-42 SAIGEQTL_0.3.5.2    data.table_1.17.8  
[4] optparse_1.7.5     

loaded via a namespace (and not attached):
[1] compiler_4.5.2        Matrix_1.7-4          tools_4.5.2          
[4] Rcpp_1.1.1 

[2026-11-23 23:00:53,051] WARNING:cellink.tl._runner: Loading required package: RhpcBLASctl
Loading required package: furrr
Loading required package: future



,CHR,POS,MarkerID,Allele1,Allele2,AC_Allele2,AF_Allele2,MissingRate,BETA,SE,Tstat,var,p.value,p.value.NA,Is.SPA,N
0,22,41900699,22_41900699_T_C,T,C,46,0.078231,0,-0.515654,0.087589,-67.2140,130.347,6.456000e-09,3.927830e-09,True,294
1,22,41901918,22_41901918_G_A,G,A,46,0.078231,0,-0.515654,0.087589,-67.2140,130.347,6.456000e-09,3.927830e-09,True,294
2,22,41901964,22_41901964_T_C,T,C,46,0.078231,0,-0.515654,0.087589,-67.2140,130.347,6.456000e-09,3.927830e-09,True,294
3,22,41902161,22_41902161_C_T,C,T,45,0.076531,0,-0.517311,0.087761,-67.1656,129.836,6.204330e-09,3.757440e-09,True,294
4,22,41902597,22_41902597_C_T,C,T,46,0.078231,0,-0.515654,0.087589,-67.2140,130.347,6.456000e-09,3.927830e-09,True,294


#### Step 3 Only: Gene-Level P-values

In [7]:
# Run only Step 3 (requires Step 2 results)
step3_results = run_saigeqtl(
    hd,
    gene_col=gene_id,
    prefix="gene1_step1",  # Same prefix
    steps=[3],  # Only Step 3
    gene_name=gene_symbol,  # Optional gene name for output
    run=True,
)

print(step3_results)

[2026-11-23 23:00:53,091] INFO:cellink.tl.external._saigeqtl: Running Step 3: Gene-level p-values


[2026-11-23 23:00:53,093] INFO:cellink.tl._runner: Executing: docker run --rm -v .:/data -v ~/cellink_data:/cellink_data -w /data wzhou88/saigeqtl /app/.pixi/envs/default/bin/Rscript /app/extdata/step3_gene_pvalue_qtl.R --assocFile=gene1_step1_results.txt --genePval_outputFile=gene1_step1_gene_pvalue.txt --geneName=SMDT1_ENSG00000183172


[2026-11-23 23:00:56,642] INFO:cellink.tl._runner: WARNING: ignoring environment value of R_HOME
R version 4.5.2 (2025-10-31)
Platform: x86_64-conda-linux-gnu
Running under: Ubuntu 20.04.6 LTS

Matrix products: default
BLAS/LAPACK: /app/.pixi/envs/default/lib/libopenblasp-r0.3.30.so;  LAPACK version 3.12.0

locale:
 [1] LC_CTYPE=C.UTF-8       LC_NUMERIC=C           LC_TIME=C.UTF-8       
 [4] LC_COLLATE=C.UTF-8     LC_MONETARY=C.UTF-8    LC_MESSAGES=C.UTF-8   
 [7] LC_PAPER=C.UTF-8       LC_NAME=C              LC_ADDRESS=C          
[10] LC_TELEPHONE=C         LC_MEASUREMENT=C.UTF-8 LC_IDENTIFICATION=C   

time zone: NA
tzcode source: system (glibc)

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
[1] RhpcBLASctl_0.23-42 SAIGEQTL_0.3.5.2    data.table_1.17.8  
[4] optparse_1.7.5     

loaded via a namespace (and not attached):
[1] compiler_4.5.2        Matrix_1.7-4          tools_4.5.2          
[4] Rcpp_1.1.1 

[2026-11-23 23:00:56,643] WARNING:cellink.tl._runner: Loading required package: RhpcBLASctl



                    gene        ACAT_p       top_MarkerID      top_pval
0  SMDT1_ENSG00000183172  1.366699e-20  22_42120602_TAA_T  2.786820e-22


### 2. Advanced: Custom Covariate Specification
Including cell-type-specific covariates and custom normalizations:

In [8]:
sc.pp.pca(hd.C, n_comps=15)


results_custom = run_saigeqtl(
    hd,
    gene_col=gene_id,
    prefix="saigeqtl_custom",
    mode="cis",
    window=cis_window,
    sample_covariates=["gPCs"],
    cell_covariates=["X_pca"],
    encode_sex=True,
    encode_age=True,
    min_mac=20,
    use_grm_to_fit_null=False,
    overwrite_variance_ratio_file=True,
    run=True,
)

results_custom.head()

[2026-11-23 23:00:59,068] INFO:cellink.io._export: Donor sex coded 0/1 is read as 0 = female, 1 = male (as in get_onek1k) and written as PLINK codes 2 (female) and 1 (male). Pass `donor_sex_map={0: 0, 1: 1}` if 0 means unknown.


[2026-11-23 23:00:59,313] INFO:cellink.io._export: Wrote PLINK 1 binary files saigeqtl_custom.bed/.bim/.fam (294 x 17569)


[2026-11-23 23:00:59,365] INFO:cellink.tl.external._saigeqtl: Running Step 1: Fitting null model


[2026-11-23 23:00:59,369] INFO:cellink.tl._runner: Executing: docker run --rm -v .:/data -v ~/cellink_data:/cellink_data -w /data wzhou88/saigeqtl /app/.pixi/envs/default/bin/Rscript /app/extdata/step1_fitNULLGLMM_qtl.R --phenoFile=saigeqtl_custom_phenotype.txt --phenoCol=ENSG00000183172 --sampleIDColinphenoFile=IND_ID --traitType=count --plinkFile=saigeqtl_custom --outputPrefix=saigeqtl_custom --useGRMtoFitNULL=FALSE --IsOverwriteVarianceRatioFile=TRUE --covarColList=sex,age,gPCs_0,gPCs_1,gPCs_2,gPCs_3,gPCs_4,gPCs_5,gPCs_6,gPCs_7,gPCs_8,gPCs_9,gPCs_10,gPCs_11,gPCs_12,gPCs_13,gPCs_14,gPCs_15,gPCs_16,gPCs_17,gPCs_18,gPCs_19,X_pca_0,X_pca_1,X_pca_2,X_pca_3,X_pca_4,X_pca_5,X_pca_6,X_pca_7,X_pca_8,X_pca_9,X_pca_10,X_pca_11,X_pca_12,X_pca_13,X_pca_14 --sampleCovarColList=sex,age,gPCs_0,gPCs_1,gPCs_2,gPCs_3,gPCs_4,gPCs_5,gPCs_6,gPCs_7,gPCs_8,gPCs_9,gPCs_10,gPCs_11,gPCs_12,gPCs_13,gPCs_14,gPCs_15,gPCs_16,gPCs_17,gPCs_18,gPCs_19


[2026-11-23 23:01:04,707] INFO:cellink.tl._runner: WARNING: ignoring environment value of R_HOME
R version 4.5.2 (2025-10-31)
Platform: x86_64-conda-linux-gnu
Running under: Ubuntu 20.04.6 LTS

Matrix products: default
BLAS/LAPACK: /app/.pixi/envs/default/lib/libopenblasp-r0.3.30.so;  LAPACK version 3.12.0

locale:
 [1] LC_CTYPE=C.UTF-8       LC_NUMERIC=C           LC_TIME=C.UTF-8       
 [4] LC_COLLATE=C.UTF-8     LC_MONETARY=C.UTF-8    LC_MESSAGES=C.UTF-8   
 [7] LC_PAPER=C.UTF-8       LC_NAME=C              LC_ADDRESS=C          
[10] LC_TELEPHONE=C         LC_MEASUREMENT=C.UTF-8 LC_IDENTIFICATION=C   

time zone: NA
tzcode source: system (glibc)

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
[1] SAIGEQTL_0.3.5.2 optparse_1.7.5  

loaded via a namespace (and not attached):
 [1] compiler_4.5.2        Matrix_1.7-4          tools_4.5.2          
 [4] Rcpp_1.1.1            getopt_1.20.4         RcppNumerical_0

[2026-11-23 23:01:04,708] WARNING:cellink.tl._runner: Loading required package: optparse
Loading required package: RhpcBLASctl
Loading required package: fastSave



[2026-11-23 23:01:04,709] INFO:cellink.tl.external._saigeqtl: Running Step 2: Association tests


[2026-11-23 23:01:04,712] INFO:cellink.tl._runner: Executing: docker run --rm -v .:/data -v ~/cellink_data:/cellink_data -w /data wzhou88/saigeqtl /app/.pixi/envs/default/bin/Rscript /app/extdata/step2_tests_qtl.R --bedFile=saigeqtl_custom.bed --bimFile=saigeqtl_custom.bim --famFile=saigeqtl_custom.fam --GMMATmodelFile=saigeqtl_custom.rda --varianceRatioFile=saigeqtl_custom.varianceRatio.txt --SAIGEOutputFile=saigeqtl_custom_results.txt --minMAC=20 --LOCO=FALSE --rangestoIncludeFile=saigeqtl_custom_ranges.txt


[2026-11-23 23:01:10,254] INFO:cellink.tl._runner: WARNING: ignoring environment value of R_HOME
R version 4.5.2 (2025-10-31)
Platform: x86_64-conda-linux-gnu
Running under: Ubuntu 20.04.6 LTS

Matrix products: default
BLAS/LAPACK: /app/.pixi/envs/default/lib/libopenblasp-r0.3.30.so;  LAPACK version 3.12.0

locale:
 [1] LC_CTYPE=C.UTF-8       LC_NUMERIC=C           LC_TIME=C.UTF-8       
 [4] LC_COLLATE=C.UTF-8     LC_MONETARY=C.UTF-8    LC_MESSAGES=C.UTF-8   
 [7] LC_PAPER=C.UTF-8       LC_NAME=C              LC_ADDRESS=C          
[10] LC_TELEPHONE=C         LC_MEASUREMENT=C.UTF-8 LC_IDENTIFICATION=C   

time zone: NA
tzcode source: system (glibc)

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
[1] RhpcBLASctl_0.23-42 SAIGEQTL_0.3.5.2    data.table_1.17.8  
[4] optparse_1.7.5     

loaded via a namespace (and not attached):
[1] compiler_4.5.2        Matrix_1.7-4          tools_4.5.2          
[4] Rcpp_1.1.1 

[2026-11-23 23:01:10,255] WARNING:cellink.tl._runner: Loading required package: RhpcBLASctl
Loading required package: furrr
Loading required package: future



,CHR,POS,MarkerID,Allele1,Allele2,AC_Allele2,AF_Allele2,MissingRate,BETA,SE,Tstat,var,p.value,p.value.NA,Is.SPA,N
0,22,41900699,22_41900699_T_C,T,C,46,0.078231,0,-0.136353,0.048694,-57.5053,421.737,0.005121,0.005107,True,294
1,22,41901918,22_41901918_G_A,G,A,46,0.078231,0,-0.136353,0.048694,-57.5053,421.737,0.005121,0.005107,True,294
2,22,41901964,22_41901964_T_C,T,C,46,0.078231,0,-0.136353,0.048694,-57.5053,421.737,0.005121,0.005107,True,294
3,22,41902161,22_41902161_C_T,C,T,45,0.076531,0,-0.137124,0.048717,-57.7762,421.343,0.004896,0.004882,True,294
4,22,41902597,22_41902597_C_T,C,T,46,0.078231,0,-0.136353,0.048694,-57.5053,421.737,0.005121,0.005107,True,294


## Advanced Usage: Dry Run and Command Generation
### Generate Commands Without Execution
Useful for debugging or running on HPC clusters:

In [9]:
# Generate command strings without executing
commands = run_saigeqtl(
    hd,
    gene_col=gene_id,
    prefix="saigeqtl_cluster",
    mode="cis",
    window=cis_window,
    sample_covariates=["gPCs"],
    use_grm_to_fit_null=False,
    overwrite_variance_ratio_file=True,
    run=False,  # Don't execute, just return commands
)

print("Step 1 (Null model):")
print(commands["step1"])
print("\nStep 2 (Association tests):")
print(commands["step2"])

[2026-11-23 23:01:12,226] INFO:cellink.io._export: Donor sex coded 0/1 is read as 0 = female, 1 = male (as in get_onek1k) and written as PLINK codes 2 (female) and 1 (male). Pass `donor_sex_map={0: 0, 1: 1}` if 0 means unknown.


[2026-11-23 23:01:12,493] INFO:cellink.io._export: Wrote PLINK 1 binary files saigeqtl_cluster.bed/.bim/.fam (294 x 17569)


Step 1 (Null model):
docker run --rm -v .:/data -v ~/cellink_data:/cellink_data -w /data wzhou88/saigeqtl /app/.pixi/envs/default/bin/Rscript /app/extdata/step1_fitNULLGLMM_qtl.R --phenoFile=saigeqtl_cluster_phenotype.txt --phenoCol=ENSG00000183172 --sampleIDColinphenoFile=IND_ID --traitType=count --plinkFile=saigeqtl_cluster --outputPrefix=saigeqtl_cluster --useGRMtoFitNULL=FALSE --IsOverwriteVarianceRatioFile=TRUE --covarColList=sex,age,gPCs_0,gPCs_1,gPCs_2,gPCs_3,gPCs_4,gPCs_5,gPCs_6,gPCs_7,gPCs_8,gPCs_9,gPCs_10,gPCs_11,gPCs_12,gPCs_13,gPCs_14,gPCs_15,gPCs_16,gPCs_17,gPCs_18,gPCs_19 --sampleCovarColList=sex,age,gPCs_0,gPCs_1,gPCs_2,gPCs_3,gPCs_4,gPCs_5,gPCs_6,gPCs_7,gPCs_8,gPCs_9,gPCs_10,gPCs_11,gPCs_12,gPCs_13,gPCs_14,gPCs_15,gPCs_16,gPCs_17,gPCs_18,gPCs_19

Step 2 (Association tests):
docker run --rm -v .:/data -v ~/cellink_data:/cellink_data -w /data wzhou88/saigeqtl /app/.pixi/envs/default/bin/Rscript /app/extdata/step2_tests_qtl.R --bedFile=saigeqtl_cluster.bed --bimFile=saigeq

## Save Commands to File
For batch submission to compute clusters:

In [10]:
commands = run_saigeqtl(
    hd,
    gene_col=gene_id,
    prefix="saigeqtl_batch",
    mode="cis",
    window=cis_window,
    sample_covariates=["gPCs"],
    run=False,
    use_grm_to_fit_null=False,
    overwrite_variance_ratio_file=True,
    save_cmd_file="saigeqtl_job.sh",
)

[2026-11-23 23:01:14,569] INFO:cellink.io._export: Donor sex coded 0/1 is read as 0 = female, 1 = male (as in get_onek1k) and written as PLINK codes 2 (female) and 1 (male). Pass `donor_sex_map={0: 0, 1: 1}` if 0 means unknown.


[2026-11-23 23:01:14,831] INFO:cellink.io._export: Wrote PLINK 1 binary files saigeqtl_batch.bed/.bim/.fam (294 x 17569)
